# batchnorm-running-stats — faded example 1: EMA-update BatchNorm1d running stats (complete the blend)

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `batchnorm-running-stats`. The last cell reports your progress on the `CNN: BatchNorm running stats` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: BatchNorm running stats` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`batchnorm-running-stats`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "batchnorm-running-stats"
DD_SUBTOPIC = "CNN: BatchNorm running stats"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

For `BatchNorm1d` on input `(B, C)`, the per-channel batch statistics reduce over the batch axis only (`dim=0`). The running buffers are then updated with the same EMA rule as 2D BatchNorm: `running = (1 - momentum) * running + momentum * batch`. The running variance uses the unbiased (`/(N-1)`) estimator, matching PyTorch's buffer convention.

## Faded exercise 1

Implement `bn1d_ema_update(x, running_mean, running_var, momentum)` for a `BatchNorm1d`-style input `x` of shape `(B, C)`. Reduce over the batch axis to get per-channel `batch_mean` and unbiased `batch_var`, then blend them into the running buffers with the EMA rule. Return `(new_running_mean, new_running_var)`. The per-channel reduction is given; you must complete the EMA blend step.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
def bn1d_ema_update(x, running_mean, running_var, momentum):
    batch_mean = x.mean(dim=0)
    batch_var = x.var(dim=0, unbiased=True)
    new_running_mean = None  # TODO: fill in this step — read the prompt cell above
    new_running_var = None   # TODO: fill in this step — read the prompt cell above
    return new_running_mean, new_running_var

t.manual_seed(0)
x = t.randn(8, 5) * 3 + 1
rm = t.zeros(5)
rv = t.ones(5)
out_mean, out_var = bn1d_ema_update(x, rm, rv, 0.1)
print("running_mean:", out_mean.round(decimals=4))
print("running_var:", out_var.round(decimals=4))


def _test():
    t.manual_seed(0)
    x = t.randn(8, 5) * 3 + 1
    rm = t.zeros(5)
    rv = t.ones(5)
    m = 0.1
    out_mean, out_var = bn1d_ema_update(x, rm, rv, m)
    bm = x.mean(dim=0)
    bv = x.var(dim=0, unbiased=True)
    exp_mean = (1 - m) * rm + m * bm
    exp_var = (1 - m) * rv + m * bv
    assert t.allclose(out_mean, exp_mean, atol=1e-6), "running_mean blend wrong"
    assert t.allclose(out_var, exp_var, atol=1e-6), "running_var blend wrong"
    # momentum=0 leaves buffers unchanged
    z_mean, z_var = bn1d_ema_update(x, rm, rv, 0.0)
    assert t.allclose(z_mean, rm, atol=1e-6) and t.allclose(z_var, rv, atol=1e-6)


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def bn1d_ema_update(x, running_mean, running_var, momentum):
    batch_mean = x.mean(dim=0)
    batch_var = x.var(dim=0, unbiased=True)
    new_running_mean = (1 - momentum) * running_mean + momentum * batch_mean
    new_running_var = (1 - momentum) * running_var + momentum * batch_var
    return new_running_mean, new_running_var

t.manual_seed(0)
x = t.randn(8, 5) * 3 + 1
rm = t.zeros(5)
rv = t.ones(5)
out_mean, out_var = bn1d_ema_update(x, rm, rv, 0.1)
print("running_mean:", out_mean.round(decimals=4))
print("running_var:", out_var.round(decimals=4))
```
</details>